# BSM L08 — AI-assisted mobile application security assessment

Pracuj na projekcie `student/apps/lesson_h_ai_security` w Android Studio.

Najpierw uruchom komórki Colab setup: sklonowanie repo z GitHuba, instalację zależności i pobranie modelu SecureBERT2.

W tym notebooku:
- H01 używa [mobsfscan](https://github.com/MobSF/mobsfscan)
- H02 używa [MobSF](https://github.com/MobSF/Mobile-Security-Framework-MobSF)
- H03 używa [SecureBERT2.0-code-vuln-detection](https://huggingface.co/cisco-ai/SecureBERT2.0-code-vuln-detection)
- H04 uruchamia w Pythonie triage między `mobsfscan` i SecureBERT2
- H05 uruchamia w Pythonie patch verifier na `SecurePatchTarget_v1` i `SecurePatchTarget_v2`

Każde zadanie ma dropdown z odpowiedzią i komórkę wysyłki.


In [ ]:
# @title Dane studenta {"run":"auto","vertical-output":true,"display-mode":"form"}
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_projektu_Kotlin = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_projektu_Kotlin,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)
# Komórka pomocnicza: formatowanie i wysyłanie odpowiedzi
answers = {}

def short_text(text, limit=48):
    text = str(text).strip().replace("\n", " ")
    return text[:limit]

def prepare_answer(*parts, limit=220):
    final_answer = "|".join(str(p) for p in parts)
    return final_answer[:limit]

def zapisz_i_wyslij(task_id, final_answer):
    final_answer = str(final_answer)
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    print(final_answer)
    # Wysyłka do backendu zadania
    wyslij_odpowiedz(task_id, final_answer)

def dropdown_choice_to_answer(choice, mapping):
    choice = str(choice).strip()
    if not choice:
        return ""
    key = choice.split(" - ", 1)[0].strip()
    return mapping.get(key, "")


In [ ]:
# @title Colab setup: clone repo from GitHub {"run":"auto","vertical-output":true,"display-mode":"form"}
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/duszekjk/mobile-systems-security.git"
CLONE_DIR = Path.cwd() / "mobile-systems-security"

def find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in candidates:
        if (base / "student/apps/lesson_h_ai_security").exists():
            return base
    for base in [CLONE_DIR, Path("/content"), Path("/content/drive/MyDrive")]:
        if base.exists():
            for candidate in [base, *base.parents]:
                if (candidate / "student/apps/lesson_h_ai_security").exists():
                    return candidate
    return None

repo_root = find_repo_root()
if repo_root is None:
    print(f"Cloning {REPO_URL}")
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
    repo_root = CLONE_DIR

print(repo_root)


In [ ]:
# @title Colab setup: install dependencies {"run":"auto","vertical-output":true,"display-mode":"form"}
!pip -q install mobsfscan transformers torch


In [ ]:
# @title Colab setup: download SecureBERT2 {"run":"auto","vertical-output":true,"display-mode":"form"}
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_id = "cisco-ai/SecureBERT2.0-code-vuln-detection"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)
print(model_id)
print(type(tokenizer).__name__)
print(type(model).__name__)


# H01 — `mobsfscan` source-code scan

## Teoria
`mobsfscan` analizuje kod źródłowy w sposób deterministyczny: szuka znanych, niebezpiecznych wzorców i dopasowuje je do reguł bezpieczeństwa. W tym labie używasz go do znalezienia hardcoded secreta, plaintext storage, nadmiernego logowania w plikach Kotlin.

Cechy tego typu narzędzia:
- daje powtarzalny wynik dla tego samego kodu,
- jest dobre do wykrywania prostych, znanych antywzorów,
- nie rozumie intencji autora aplikacji,
- nie zastępuje ręcznej weryfikacji, jeśli finding jest niejednoznaczny.

## Co sprawdzasz
Masz przejść katalog `student/apps/lesson_h_ai_security/InsecureNotes/app/src/main/java/com/example/secretlab/insecure/` narzędziem `mobsfscan` i znaleźć przygotowaną podatność w kodzie źródłowym. To zadanie uczy, jak odczytać finding ze skanera i przypisać go do jednego kanonicznego kodu.

## Narzędzie
Jeśli nie masz narzędzia, zainstaluj je w terminalu projektu poleceniem `python3 -m pip install mobsfscan`. Dokumentacja i źródła są na stronie [MobSF/mobsfscan](https://github.com/MobSF/mobsfscan). Potem uruchom `mobsfscan` z katalogu `student/apps/lesson_h_ai_security/InsecureNotes`.

## Krok po kroku
1. Otwórz `student/apps/lesson_h_ai_security/InsecureNotes` w Android Studio.
1. Otwórz terminal w katalogu projektu.
1. Uruchom `mobsfscan app/src/main/java/com/example/secretlab/insecure/`.
1. Odczytaj finding dla `SecretStore.kt`, `LoggingHelper.kt` i `CryptoHelper.kt`.
1. Jeśli masz kilka wyników, wybierz ten z najwyższym znaczeniem bezpieczeństwa: sekret na sztywno, plaintext storage, ekspozycja danych w logach.
1. Jeśli raport pokazuje kilka problemów, wybierz ten, który dotyczy sekretnych stałych w kodzie i ich zapisu w zwykłym storage.
1. Wybierz w dropdownie odpowiedź, która najlepiej opisuje wykryty problem.

## Weryfikacja
W formularzu wybierz jedną odpowiedź odpowiadającą findingowi. Notebook sam zamieni wybór na kanoniczny kod.


In [ ]:
# @title H01 — Formularz odpowiedzi {"run":"auto","vertical-output":true,"single-column":true,"display-mode":"form"}
choice_h01 = ""  #@param ["", "HARDCODED_SECRET - hardcoded secret literal in SecretStore.kt", "PLAINTEXT_SHARED_PREFERENCES - plaintext token in SharedPreferences", "SENSITIVE_LOGGING - sensitive logging in LoggingHelper.kt", "WEAK_CRYPTO - weak crypto in CryptoHelper.kt", "EXPORTED_COMPONENT - exported component in manifest"]
final_answer = prepare_answer(dropdown_choice_to_answer(choice_h01, {"HARDCODED_SECRET": "HARDCODED_SECRET", "PLAINTEXT_SHARED_PREFERENCES": "PLAINTEXT_SHARED_PREFERENCES", "SENSITIVE_LOGGING": "SENSITIVE_LOGGING", "WEAK_CRYPTO": "WEAK_CRYPTO", "EXPORTED_COMPONENT": "EXPORTED_COMPONENT"}))
print(final_answer)
zapisz_i_wyslij("H01", final_answer)


# H02 — APK analysis in MobSF

## Teoria
MobSF czyta już zbudowany APK. W tym zadaniu nie patrzysz na kod źródłowy, tylko na to, co znalazło się w paczce: manifest, flagi aplikacji, komponenty eksportowane, konfigurację backupu i cleartext traffic. Raport MobSF jest zbudowany z sekcji, które pomagają szybko wyłapać słabą konfigurację bezpieczeństwa.

Cechy MobSF w tym labie:
- analizuje artefakt po buildzie, a nie pliki robocze w projekcie,
- pokazuje wynik manifestu i flag bezpieczeństwa,
- jest użyteczny do szybkiej oceny konfiguracji APK,
- nie zastępuje porównania z wersją źródłową, jeśli raport jest niepełny.

## Co sprawdzasz
Masz przeanalizować APK `FakeBankLite` w MobSF i wskazać problem widoczny w raporcie manifestu. W tym zadaniu chodzi konkretnie o ustawienie `android:usesCleartextTraffic`.

## Narzędzie
Użyj [MobSF](https://github.com/MobSF/Mobile-Security-Framework-MobSF). To webowa aplikacja z dashboardem, w którym klikniesz `Upload & Analyze`. Jeśli nie widzisz panelu uploadu, to nie jest właściwa instancja narzędzia.

## Krok po kroku
1. Otwórz `student/apps/lesson_h_ai_security/FakeBankLite` w Android Studio.
1. W terminalu w katalogu projektu uruchom `./gradlew assembleDebug`.
1. Otwórz instancję MobSF w przeglądarce i kliknij `Upload & Analyze`.
1. Wgraj plik `app/build/outputs/apk/debug/app-debug.apk`.
1. Poczekaj, aż raport się wygeneruje, i otwórz `Manifest Analysis`.
1. Odszukaj linię `android:usesCleartextTraffic="true"`.
1. Wybierz odpowiedź odnoszącą się do cleartext traffic, nie do backupu ani debugowania.

## Weryfikacja
W formularzu wybierz jedną odpowiedź odpowiadającą dokładnie temu ustawieniu manifestu. Notebook sam zamieni wybór na kanoniczny kod.


In [ ]:
# @title H02 — Formularz odpowiedzi {"run":"auto","vertical-output":true,"single-column":true,"display-mode":"form"}
choice_h02 = ""  #@param ["", "CLEARTEXT_TRAFFIC - android:usesCleartextTraffic=\"true\" in manifest", "BACKUP_ENABLED - android:allowBackup=\"true\" in manifest", "DEBUGGABLE_TRUE - android:debuggable=\"true\" in manifest", "EXPORTED_COMPONENT - exported launcher activity", "NO_ISSUE - network security config present"]
final_answer = prepare_answer(dropdown_choice_to_answer(choice_h02, {"CLEARTEXT_TRAFFIC": "CLEARTEXT_TRAFFIC", "BACKUP_ENABLED": "BACKUP_ENABLED", "DEBUGGABLE_TRUE": "DEBUGGABLE_TRUE", "EXPORTED_COMPONENT": "EXPORTED_COMPONENT", "NO_ISSUE": "NO_ISSUE"}))
print(final_answer)
zapisz_i_wyslij("H02", final_answer)


# H03 — `SecureBERT2` security classification

## Teoria
SecureBERT2 bierze pełny plik źródłowy, zamienia go na tokeny i zwraca label klasyfikacyjny. To nie jest regułowy skaner. Model ocenia wzorce w tekście kodu i potrafi zareagować na podobieństwo do znanych podatności, ale jego wynik zależy od dokładnego wejścia, wersji modelu i sposobu przygotowania pliku.

## Uruchomienie
Poniższa komórka kodu czyta cały plik `SecretStore.kt` i przekazuje go do SecureBERT2.

## Krok po kroku
1. Otwórz `student/apps/lesson_h_ai_security/InsecureNotes/app/src/main/java/com/example/secretlab/insecure/SecretStore.kt`.
1. Uruchom komórkę kodu poniżej.
1. Odczytaj label modelu i sprawdź, czy model widzi podatność w pełnym pliku.
1. Wybierz odpowiedź z dropdownu.

## Weryfikacja
W formularzu wybierasz odpowiedź odpowiadającą labelowi modelu. Notebook sam zamieni wybór na kanoniczny kod.


In [ ]:
# H03 — SecureBERT2 full-file classifier
from pathlib import Path
from transformers import pipeline

file_path = repo_root / "student/apps/lesson_h_ai_security/InsecureNotes/app/src/main/java/com/example/secretlab/insecure/SecretStore.kt"
full_source = file_path.read_text()
classifier = pipeline("text-classification", model="cisco-ai/SecureBERT2.0-code-vuln-detection")
result = classifier(full_source, truncation=True)[0]
print(file_path)
print(result)


In [ ]:
# @title H03 — Formularz odpowiedzi {"run":"auto","vertical-output":true,"single-column":true,"display-mode":"form"}
choice_h03 = ""  #@param ["", "MODEL_DETECTED_VULNERABLE - SecureBERT2 flags the full file as vulnerable", "MODEL_RESULT_SAFE - SecureBERT2 says the file is safe", "MODEL_RESULT_UNCLEAR - SecureBERT2 returns an unclear result", "MODEL_INPUT_INVALID - input was invalid", "MODEL_TOOL_UNAVAILABLE - tool unavailable"]
final_answer = prepare_answer(dropdown_choice_to_answer(choice_h03, {"MODEL_DETECTED_VULNERABLE": "MODEL_DETECTED_VULNERABLE", "MODEL_RESULT_SAFE": "MODEL_RESULT_SAFE", "MODEL_RESULT_UNCLEAR": "MODEL_RESULT_UNCLEAR", "MODEL_INPUT_INVALID": "MODEL_INPUT_INVALID", "MODEL_TOOL_UNAVAILABLE": "MODEL_TOOL_UNAVAILABLE"}))
print(final_answer)
zapisz_i_wyslij("H03", final_answer)


# H04 — Heterogeneous multi-agent disagreement triage

## Teoria
Heterogeneous multi-agent disagreement triage to układ, w którym regułowy skaner, model klasyfikacyjny i lokalny verifier analizują ten sam pełny plik kodu i zwracają własne sygnały.

## Uruchomienie
Poniższa komórka kodu uruchamia `mobsfscan` oraz SecureBERT2 na tym samym pełnym pliku `SecretStore.kt` i wypisuje oba wyniki.

## Co sprawdzasz
Porównujesz wynik `mobsfscan` z H01 i wynik SecureBERT2 z tego samego pełnego pliku `SecretStore.kt`. Jeśli skaner wskazuje hardcoded secret, a model zwraca wynik bez podatności, wybierasz kanoniczny kod.

## Weryfikacja
Notebook zapisuje kanoniczny kod `HETEROGENEOUS_MULTI_AGENT_TRIAGE`.


In [ ]:
# H04 — multi-agent triage on full file
import subprocess
from pathlib import Path
from transformers import pipeline

file_path = repo_root / "student/apps/lesson_h_ai_security/InsecureNotes/app/src/main/java/com/example/secretlab/insecure/SecretStore.kt"
full_source = file_path.read_text()

scanner_root = repo_root / "student/apps/lesson_h_ai_security/InsecureNotes"
scanner = subprocess.run(["mobsfscan", "app/src/main/java/com/example/secretlab/insecure/"], cwd=str(scanner_root), capture_output=True, text=True)
classifier = pipeline("text-classification", model="cisco-ai/SecureBERT2.0-code-vuln-detection")
model_result = classifier(full_source, truncation=True)[0]
print(file_path)
print(scanner.stdout)
print(model_result)


In [ ]:
# @title H04 — Formularz odpowiedzi {"run":"auto","vertical-output":true,"single-column":true,"display-mode":"form"}
choice_h04 = ""  #@param ["", "HETEROGENEOUS_MULTI_AGENT_TRIAGE - multi-agent verifier flags scanner/model disagreement", "SCANNER_ONLY - scanner only", "MODEL_ONLY - model only", "MANUAL_REVIEW_NEEDED - manual review only"]
final_answer = prepare_answer(dropdown_choice_to_answer(choice_h04, {"HETEROGENEOUS_MULTI_AGENT_TRIAGE": "HETEROGENEOUS_MULTI_AGENT_TRIAGE", "SCANNER_ONLY": "SCANNER_ONLY", "MODEL_ONLY": "MODEL_ONLY", "MANUAL_REVIEW_NEEDED": "MANUAL_REVIEW_NEEDED"}))
print(final_answer)
zapisz_i_wyslij("H04", final_answer)


# H05 — Agentic patch verification

## Teoria
Agentic patch verification to uruchomiony w Pythonie verifier, który porównuje dwa pełne manifesty przed i po poprawce, wyciąga zmienione linie i sprawdza, czy patch usuwa ryzyko `usesCleartextTraffic`.

## Uruchomienie
Poniższa komórka kodu czyta dwa pełne manifesty, liczy diff i przekazuje go do verifiera.

## Co sprawdzasz
Podajesz diff do verifiera i sprawdzasz, czy zmiana `android:usesCleartextTraffic="true"` na `android:usesCleartextTraffic="false"` usuwa problem.

## Weryfikacja
Notebook zapisuje kanoniczny kod `AGENTIC_PATCH_VERIFICATION`.


In [ ]:
# H05 — agentic patch verification on full manifests
from pathlib import Path
from difflib import unified_diff
from transformers import pipeline

before_path = repo_root / "student/apps/lesson_h_ai_security/SecurePatchTarget_v1/app/src/main/AndroidManifest.xml"
after_path = repo_root / "student/apps/lesson_h_ai_security/SecurePatchTarget_v2/app/src/main/AndroidManifest.xml"
before = before_path.read_text()
after = after_path.read_text()
diff = "\n".join(unified_diff(before.splitlines(), after.splitlines(), fromfile=str(before_path), tofile=str(after_path)))
verifier = pipeline("text-classification", model="cisco-ai/SecureBERT2.0-code-vuln-detection")
print(diff)
print(verifier(diff, truncation=True)[0])


In [ ]:
# @title H05 — Formularz odpowiedzi {"run":"auto","vertical-output":true,"single-column":true,"display-mode":"form"}
choice_h05 = ""  #@param ["", "AGENTIC_PATCH_VERIFICATION - patch verifier confirms cleartext is removed", "BACKUP_DISABLED - backup disabled", "DEBUGGABLE_REMOVED - debuggable removed", "EXPORTED_COMPONENT_REMOVED - exported component removed", "NO_FIX_DETECTED - no visible change"]
final_answer = prepare_answer(dropdown_choice_to_answer(choice_h05, {"AGENTIC_PATCH_VERIFICATION": "AGENTIC_PATCH_VERIFICATION", "BACKUP_DISABLED": "BACKUP_DISABLED", "DEBUGGABLE_REMOVED": "DEBUGGABLE_REMOVED", "EXPORTED_COMPONENT_REMOVED": "EXPORTED_COMPONENT_REMOVED", "NO_FIX_DETECTED": "NO_FIX_DETECTED"}))
print(final_answer)
zapisz_i_wyslij("H05", final_answer)
